In [9]:
import pandas as pd
import psycopg
from psycopg import sql
import os
from dotenv import load_dotenv
from pathlib import Path


In [10]:
RAIZ = Path.cwd().parent
load_dotenv(RAIZ / ".env")
db = os.getenv('DATABASE_URL')

docs = RAIZ / "docs"
docs.mkdir(exist_ok=True)
print(docs / "profiling.csv")

D:\bases de datos\secop-pipeline\docs\profiling.csv


In [11]:
query = sql.SQL("""
            SELECT
            count(*)                                                          AS total,
            count(*) FILTER (WHERE {c} IS NULL)       AS nulls,
            count(*) FILTER (WHERE lower(trim({c}))
                             IN ('', 'no definido'))                          AS empty_pseudonulls,
            count(DISTINCT {c})                       AS distinct_count,
            max(length({c}))                          AS length_max
            FROM raw.contratos_electronicos;
         """)

In [12]:
# 1. columnas de raw


with psycopg.connect(db) as conn:

    cols = [r[0] for r in conn.execute("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema = 'raw' AND table_name = 'contratos_electronicos'
    ORDER BY ordinal_position""").fetchall()]

    results = []
    for c in cols:

        row = conn.execute(query.format(c=sql.Identifier(c))).fetchone()
        results.append((c, *row))

profiling = pd.DataFrame(results, columns=['columns', 'total', 'nulls', 'empty_pseudonulls', 'distinct_count', 'length_max'])

profiling["pct_nulos"] = (
    (profiling["nulls"] + profiling["empty_pseudonulls"]) / profiling["total"] * 100
).round(2)



# 3. guardar
profiling.to_csv(docs / "profiling.csv", index=False)